# 026 — Late Chunking
**Chunking Series** | Jina AI 2024: embed the full document first, chunk embeddings later

Covers: Standard chunking problem · Token-level embeddings · Mean-pool per chunk · Retrieval comparison


In [1]:
import os, warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', module="langchain.*")
warnings.filterwarnings('ignore', module="langgraph.*")

from dotenv import load_dotenv
load_dotenv(os.path.join(os.getcwd(), '..', '.env'))
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
assert GROQ_API_KEY, "GROQ_API_KEY not found — check ../.env"
print("✓ API key loaded")


✓ API key loaded


In [2]:
import os
os.environ["ANONYMIZED_TELEMETRY"] = "False"

from langchain_groq import ChatGroq
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0, max_retries=2)
llm_smart = ChatGroq(model="llama-3.3-70b-versatile", temperature=0, max_retries=2)
print(f"✓ llm       : {llm.model_name}")
print(f"✓ llm_smart : {llm_smart.model_name}")


✓ llm       : llama-3.1-8b-instant
✓ llm_smart : llama-3.3-70b-versatile


In [3]:
import os
DS_DIR = os.path.join(os.getcwd(), '..', 'datasets')

def load_doc(fname):
    path = os.path.join(DS_DIR, fname)
    return open(path, encoding='utf-8').read()

ai_text  = load_doc("ai.txt")
ml_text  = load_doc("machine_learning.txt")
nlp_text = load_doc("nlp.txt")
rag_text = load_doc("rag.txt")
dl_text  = load_doc("deep_learning.txt")
all_text = "\n\n".join([ai_text, ml_text, nlp_text, rag_text])
print(f"✓ Loaded {len(all_text):,} chars")


✓ Loaded 22,486 chars


In [4]:
# ── 1. The problem with standard chunk-then-embed ───────────────────────
print("STANDARD (chunk-then-embed):")
print("  1. Split document into chunks")
print("  2. Embed each chunk independently")
print("  Problem: each chunk loses its surrounding context.")
print()
print("Example from rag_text:")
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=0)
chunks = splitter.split_text(rag_text[:2000])
print(f"  Chunk 3: {repr(chunks[3][:120])}")
print()
print("  The word 'it' or 'this approach' in chunk 3 has no context.")
print("  The embedding of chunk 3 cannot know what 'it' refers to.")
print()
print("LATE CHUNKING (Jina AI 2024):")
print("  1. Pass FULL document through transformer → get token embeddings")
print("  2. Split token embeddings at chunk boundaries → mean pool each part")
print("  Each chunk embedding retains full-document context from attention.")


STANDARD (chunk-then-embed):
  1. Split document into chunks
  2. Embed each chunk independently
  Problem: each chunk loses its surrounding context.

Example from rag_text:
  Chunk 3: 'RAG was introduced by Lewis et al. (2020) as a way to give language models access to\nup-to-date information and verifiab'

  The word 'it' or 'this approach' in chunk 3 has no context.
  The embedding of chunk 3 cannot know what 'it' refers to.

LATE CHUNKING (Jina AI 2024):
  1. Pass FULL document through transformer → get token embeddings
  2. Split token embeddings at chunk boundaries → mean pool each part
  Each chunk embedding retains full-document context from attention.


In [5]:
# ── 2. Extract token embeddings from the full document ──────────────────
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel

MODEL = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(
    MODEL,
    cache_dir=os.path.join(os.getcwd(), '..', 'datasets', 'models')
)
model = AutoModel.from_pretrained(
    MODEL,
    cache_dir=os.path.join(os.getcwd(), '..', 'datasets', 'models')
)
model.eval()
print(f"✓ Model loaded: {MODEL}")
print(f"  Max tokens: {tokenizer.model_max_length}")

# Use a short passage for demo (model max = 512 tokens)
# In production use jina-embeddings-v2-base-en (8192 tokens)
passage = rag_text[:1200]
tokens = tokenizer(passage, return_tensors="pt", truncation=True, max_length=512)
print(f"  Passage tokens: {tokens['input_ids'].shape[1]}")

with torch.no_grad():
    output = model(**tokens)

# output.last_hidden_state: [1, seq_len, hidden_dim]
token_embeddings = output.last_hidden_state[0].numpy()  # [seq_len, 384]
print(f"  Token embeddings shape: {token_embeddings.shape}")


✓ Model loaded: sentence-transformers/all-MiniLM-L6-v2
  Max tokens: 512
  Passage tokens: 235
  Token embeddings shape: (235, 384)


In [6]:
# ── 3. Map chunk boundaries to token positions ──────────────────────────
# Find where each text chunk starts/ends in the token sequence

def get_chunk_token_spans(text, chunk_texts, tokenizer):
    spans = []
    char_pos = 0
    full_ids = tokenizer(text, return_tensors="pt", truncation=True,
                         max_length=512)['input_ids'][0].tolist()
    for chunk in chunk_texts:
        chunk_ids = tokenizer(chunk, add_special_tokens=False)['input_ids']
        # Find chunk_ids as a subsequence in full_ids
        n, m = len(full_ids), len(chunk_ids)
        start = None
        for i in range(n - m + 1):
            if full_ids[i:i+m] == chunk_ids:
                start = i
                break
        if start is not None:
            spans.append((start, start + m))
        else:
            spans.append(None)
    return spans

# Re-chunk the passage
chunks_lc = splitter.split_text(passage)[:6]
spans = get_chunk_token_spans(passage, chunks_lc, tokenizer)
for i, (chunk, span) in enumerate(zip(chunks_lc, spans)):
    status = f"tokens {span[0]}-{span[1]}" if span else "not found"
    print(f"Chunk {i}: [{status}] {repr(chunk[:60])}")


Chunk 0: [tokens 1-8] 'Retrieval-Augmented Generation (RAG)'
Chunk 1: [tokens 8-37] 'Retrieval-Augmented Generation (RAG) is a hybrid AI architec'
Chunk 2: [tokens 37-58] 'parameters, RAG systems retrieve relevant documents from an '
Chunk 3: [tokens 58-97] 'RAG was introduced by Lewis et al. (2020) as a way to give l'
Chunk 4: [tokens 97-120] '(weights-only) language models: knowledge staleness, halluci'
Chunk 5: [tokens 120-132] 'Core RAG Pipeline\n\nA standard RAG pipeline consists of three'


In [7]:
# ── 4. Mean-pool token embeddings per chunk (late chunking) ─────────────
def late_chunk_embed(token_embeddings, spans):
    chunk_vecs = []
    for span in spans:
        if span is None:
            chunk_vecs.append(None)
            continue
        start, end = span
        # +1 for [CLS] offset, clip to available tokens
        start_idx = min(start + 1, len(token_embeddings) - 1)
        end_idx   = min(end   + 1, len(token_embeddings))
        if start_idx >= end_idx:
            end_idx = start_idx + 1
        chunk_vec = token_embeddings[start_idx:end_idx].mean(axis=0)
        chunk_vec = chunk_vec / (np.linalg.norm(chunk_vec) + 1e-8)
        chunk_vecs.append(chunk_vec)
    return chunk_vecs

late_vecs = late_chunk_embed(token_embeddings, spans)
print(f"Late chunk embeddings: {sum(v is not None for v in late_vecs)}/{len(spans)} produced")
print(f"Embedding dimension: {late_vecs[0].shape if late_vecs[0] is not None else 'N/A'}")


Late chunk embeddings: 6/6 produced
Embedding dimension: (384,)


In [8]:
# ── 5. Standard embeddings for comparison ───────────────────────────────
from langchain_huggingface import HuggingFaceEmbeddings

std_embedder = HuggingFaceEmbeddings(
    model_name=MODEL,
    cache_folder=os.path.join(os.getcwd(), '..', 'datasets', 'models'),
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)
std_vecs = std_embedder.embed_documents(chunks_lc)
std_vecs = [np.array(v) for v in std_vecs]
print(f"Standard embeddings: {len(std_vecs)} vectors of dim {std_vecs[0].shape[0]}")


Standard embeddings: 6 vectors of dim 384


In [9]:
# ── 6. Retrieval comparison: standard vs late chunking ───────────────────
def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8))

queries = [
    "How does retrieval augmented generation work?",
    "vector database similarity search",
    "reduce hallucination in language models",
]

for q in queries:
    q_vec = np.array(std_embedder.embed_query(q))
    print(f"\nQuery: {q!r}")
    for i, (chunk, std_v, late_v) in enumerate(zip(chunks_lc, std_vecs, late_vecs)):
        if late_v is None:
            continue
        s_score = cosine(q_vec, std_v)
        l_score = cosine(q_vec, late_v)
        diff = l_score - s_score
        marker = "  ▲" if diff > 0.01 else ("  ▼" if diff < -0.01 else "   ")
        print(f"  Chunk {i}: std={s_score:.3f}  late={l_score:.3f}{marker}  |  {chunk[:60]!r}")



Query: 'How does retrieval augmented generation work?'
  Chunk 0: std=0.822  late=0.667  ▼  |  'Retrieval-Augmented Generation (RAG)'
  Chunk 1: std=0.644  late=0.600  ▼  |  'Retrieval-Augmented Generation (RAG) is a hybrid AI architec'
  Chunk 2: std=0.363  late=0.394  ▲  |  'parameters, RAG systems retrieve relevant documents from an '
  Chunk 3: std=0.234  late=0.448  ▲  |  'RAG was introduced by Lewis et al. (2020) as a way to give l'
  Chunk 4: std=0.209  late=0.388  ▲  |  '(weights-only) language models: knowledge staleness, halluci'
  Chunk 5: std=0.156  late=0.318  ▲  |  'Core RAG Pipeline\n\nA standard RAG pipeline consists of three'

Query: 'vector database similarity search'
  Chunk 0: std=0.277  late=0.292  ▲  |  'Retrieval-Augmented Generation (RAG)'
  Chunk 1: std=0.293  late=0.373  ▲  |  'Retrieval-Augmented Generation (RAG) is a hybrid AI architec'
  Chunk 2: std=0.242  late=0.284  ▲  |  'parameters, RAG systems retrieve relevant documents from an '
  Chunk 3: std=0.14

In [10]:
# ── 7. Summary: when to use late chunking ───────────────────────────────
print("LATE CHUNKING — KEY POINTS")
print("=" * 45)
print()
print("How it works:")
print("  1. Tokenize FULL document (needs long-context model)")
print("  2. Run transformer → get per-token embeddings")
print("  3. Determine chunk boundaries in token space")
print("  4. Mean-pool token embeddings within each chunk boundary")
print("  5. Use resulting vectors for indexing")
print()
print("Why it works:")
print("  Transformer attention is computed over the full document.")
print("  Every token embedding already 'knows' about surrounding tokens.")
print("  Pooling those token embeddings into chunk vectors preserves context.")
print()
print("Limitation of this demo:")
print("  all-MiniLM-L6-v2 max = 512 tokens (~380 words).")
print("  For real late chunking use jina-embeddings-v2-base-en (8192 tokens).")
print("  pip install -U transformers  then use 'jinaai/jina-embeddings-v2-base-en'")
print()
print("vs Contextual Retrieval (notebook 021):")
print("  Contextual retrieval: LLM writes a context prefix for each chunk (LLM call per chunk)")
print("  Late chunking      : No LLM call. Context comes from the embedding model itself.")


LATE CHUNKING — KEY POINTS

How it works:
  1. Tokenize FULL document (needs long-context model)
  2. Run transformer → get per-token embeddings
  3. Determine chunk boundaries in token space
  4. Mean-pool token embeddings within each chunk boundary
  5. Use resulting vectors for indexing

Why it works:
  Transformer attention is computed over the full document.
  Every token embedding already 'knows' about surrounding tokens.
  Pooling those token embeddings into chunk vectors preserves context.

Limitation of this demo:
  all-MiniLM-L6-v2 max = 512 tokens (~380 words).
  For real late chunking use jina-embeddings-v2-base-en (8192 tokens).
  pip install -U transformers  then use 'jinaai/jina-embeddings-v2-base-en'

vs Contextual Retrieval (notebook 021):
  Contextual retrieval: LLM writes a context prefix for each chunk (LLM call per chunk)
  Late chunking      : No LLM call. Context comes from the embedding model itself.
